### **Importation**

In [ ]:
# Utilitaires de base
import builtins
import pandas as pd

# Suivi des expriences (MLflow & DagsHub)
import dagshub
import mlflow

# Scikit-Learn : Sparation des donnes et mtriques d'valuation
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, 
    classification_report, 
    f1_score, 
    fbeta_score, 
    precision_score, 
    recall_score
)

# TensorFlow / Keras : Cration et entranement du modle Deep Learning
import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.layers import TextVectorization


#### **DagsHub & MLflow Init**

In [ ]:
# Initialisation de la connexion DagsHub avec les identifiants de votre dépôt et activation du mode MLflow
# PATCH WINDOWS : Force l'utilisation de l'encodage UTF-8 lors de l'écriture des fichiers.

_original_open = builtins.open
def _utf8_open(*args, **kwargs):
    mode = kwargs.get('mode', args[1] if len(args) > 1 else 'r')
    if 'b' not in mode and 'encoding' not in kwargs:
        kwargs['encoding'] = 'utf-8'
    return _original_open(*args, **kwargs)
builtins.open = _utf8_open

dagshub.init(repo_owner='Oscar-AS', repo_name='disaster-tweets-project', mlflow=True)

# Définition du nom du dossier (expérience) dans MLflow où toutes nos métriques seront classées
mlflow.set_experiment("Disaster_Tweets")

# Affichage d'un message console pour confirmer que le tracking est bien connecté
print("MLflow activé avec succès sur DagsHub !")


Initialized MLflow to track repo "Oscar-AS/disaster-tweets-project"

Repository Oscar-AS/disaster-tweets-project initialized!

MLflow activé avec succès sur DagsHub !


#### **Importation Données**

In [ ]:
#Chargement des données
df = pd.read_csv("Base/tweets_clean.csv")

### **Séparation des données**

In [ ]:

# Séparation Train/Test (80% / 20%)
X_train, X_test, y_train, y_test = train_test_split(
    df, df['target'], test_size=0.2, random_state=42, stratify=df['target']
)

# Affiche dans la console le nombre de tweets utilisés pour l'entraînement
print(f"Taille de l'entraînement : {len(X_train)}")
# Affiche dans la console le nombre de tweets gardés pour le test
print(f"Taille du test : {len(X_test)}")


#### **Implémentation des modèles (Réseaux de Neurones avec TensorFlow/Keras)**

In [ ]:
# Définition de la taille maximale du vocabulaire autorisé (les 15000 mots les plus fréquents)
MAX_VOCAB_SIZE = 15000
# Définition de la taille maximale d'une phrase (tronquée si plus longue, remplie par des 0 si plus courte)
MAX_SEQUENCE_LENGTH = 128

# Instanciation de la couche de Vectorisation
vectorizer = TextVectorization(
    max_tokens=MAX_VOCAB_SIZE, # Limite du vocabulaire
    output_mode='int',         # Chaque mot sera remplacé par un nombre entier
    output_sequence_length=MAX_SEQUENCE_LENGTH # Fixe la longueur de toutes les séquences à 128
)

# Apprentissage du vocabulaire : on lit le texte d'entraînement pour créer le dictionnaire mot -> entier
vectorizer.adapt(X_train.to_numpy())

# Fonction utilitaire pour préparer les données afin que TensorFlow s'entraîne plus vite
def prepare_tf_dataset(X, y, batch_size=32):
    # Création d'un dataset TensorFlow à partir de nos listes Python (X et y)
    dataset = tf.data.Dataset.from_tensor_slices((X, y))
    # Groupement des données en paquets (batches) de 32, et mise en mémoire cache dynamique (AUTOTUNE)
    dataset = dataset.batch(batch_size).prefetch(tf.data.AUTOTUNE)
    # Retourne le dataset optimisé
    return dataset

# Création du Dataset d'entraînement accéléré
train_ds = prepare_tf_dataset(X_train, y_train)
# Création du Dataset de test accéléré
test_ds = prepare_tf_dataset(X_test, y_test)

# Importation du module MLflow dédié à TensorFlow
# Activation du suivi automatique (enregistrera la loss, les paramètres et les modèles à chaque epoch sans coder manuellement)
mlflow.tensorflow.autolog(log_models=True)


#### **Modèle TextCNN (Convolutional Neural Network 1D)**

##### **Description du modèle**
Historiquement inventé pour l'analyse d'images (pour détecter des bords, des formes), le CNN a été adapté pour le texte (TextCNN) avec un succès retentissant.

##### **Explication du fonctionnement**
Au lieu de lire mot par mot, le CNN utilise des "fenêtres glissantes" (Filtres Convolutifs) qui regardent des groupes de 3, 4 ou 5 mots à la fois. Le modèle cherche spécifiquement des **"motifs" ou "expressions clés"** (ex: "building on fire", "heavy earthquake") indépendamment de leur position dans la phrase.


In [17]:
# Démarrage d'un nouveau Run MLflow nommé "3.2_TextCNN"
with mlflow.start_run(run_name="3.2_TextCNN"):
    # Création d'un nouveau modèle en couches empilées
    model_cnn = models.Sequential([
        # Couche 1 : Vectorisation (texte vers entiers)
        vectorizer,
        # Couche 2 : Embedding (entiers vers vecteurs mathématiques)
        layers.Embedding(input_dim=MAX_VOCAB_SIZE, output_dim=64),
        # Couche 3 : Convolution 1D, qui utilise 64 filtres et lit des paquets de 5 mots (kernel_size=5) avec une fonction d'activation ReLU
        layers.Conv1D(filters=64, kernel_size=5, activation='relu'),
        # Couche 4 : Regroupement Max Global (ne garde que l'information la plus importante détectée par la convolution)
        layers.GlobalMaxPooling1D(),
        # Couche 5 : Couche dense de 32 neurones pour interpréter l'information
        layers.Dense(32, activation='relu'),
        # Couche 6 : Dropout (désactive aléatoirement 50% des neurones pour éviter le surapprentissage)
        layers.Dropout(0.5),
        # Couche 7 : Prédiction binaire finale (1 neurone, Sigmoid)
        layers.Dense(1, activation='sigmoid')
    ])
    
    # Préparation du modèle avec l'optimiseur adam et suivi de l'accuracy
    model_cnn.compile(loss='binary_crossentropy', optimizer='adam', metrics=['accuracy'])
    # Entraînement pendant 3 epochs
    model_cnn.fit(train_ds, validation_data=test_ds, epochs=3)
    
    # Récupération des prédictions formatées en 0 ou 1
    y_pred_cnn = (model_cnn.predict(test_ds) > 0.5).astype(int)
    
    # Affichage dans la console
    print("\n--- Rapport TextCNN ---")
    # Impression du rapport final (F1, Precision, Recall)
    print(classification_report(y_test, y_pred_cnn))
    
    # Suivi explicite des métriques de test dans MLflow
    mlflow.log_metric("eval_f1_macro", f1_score(y_test, y_pred_cnn, average='macro'))
    mlflow.log_metric("eval_f2_score", fbeta_score(y_test, y_pred_cnn, beta=2, average='macro'))
    
    precision_cls = precision_score(y_test, y_pred_cnn, average=None)
    recall_cls = recall_score(y_test, y_pred_cnn, average=None)
    mlflow.log_metric("eval_precision_class_0", precision_cls[0])
    mlflow.log_metric("eval_precision_class_1", precision_cls[1])
    mlflow.log_metric("eval_recall_class_0", recall_cls[0])
    mlflow.log_metric("eval_recall_class_1", recall_cls[1])
    mlflow.log_metric("eval_accuracy", accuracy_score(y_test, y_pred_cnn))
    
    # Enregistrement explicite du modèle pour la mise en production
    try:
        mlflow.tensorflow.log_model(model_cnn, "model")
    except Exception as e:
        print("Erreur lors de la sauvegarde du modèle Keras :", e)


2026/05/06 11:42:08 WARNING mlflow.tensorflow: Encountered unexpected error while inferring batch size from training dataset: Sequential model 'sequential_7' has no defined input shape yet.
2026/05/06 11:42:09 WARNING mlflow.tensorflow: Failed to log training dataset information to MLflow Tracking. Reason: 'ascii' codec can't decode byte 0xe2 in position 120: ordinal not in range(128)


Epoch 1/3
283/285 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - accuracy: 0.8133 - loss: 0.5034

285/285 ━━━━━━━━━━━━━━━━━━━━ 31s 92ms/step - accuracy: 0.8311 - loss: 0.4350 - val_accuracy: 0.8747 - val_loss: 0.3055
Epoch 2/3
285/285 ━━━━━━━━━━━━━━━━━━━━ 6s 21ms/step - accuracy: 0.9148 - loss: 0.2310 - val_accuracy: 0.8843 - val_loss: 0.3178
Epoch 3/3
285/285 ━━━━━━━━━━━━━━━━━━━━ 10s 33ms/step - accuracy: 0.9642 - loss: 0.1093 - val_accuracy: 0.8839 - val_loss: 0.3862


2026/05/06 11:42:57 WARNING mlflow.tensorflow: Failed to infer model signature: Invalid dtype: object
2026/05/06 11:42:57 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:43:01 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:43:22 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmpb4ax6h_r\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


72/72 ━━━━━━━━━━━━━━━━━━━━ 1s 11ms/step

--- Rapport TextCNN ---
              precision    recall  f1-score   support

           0       0.91      0.95      0.93      1851
           1       0.73      0.59      0.65       423

    accuracy                           0.88      2274
   macro avg       0.82      0.77      0.79      2274
weighted avg       0.88      0.88      0.88      2274



2026/05/06 11:43:44 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/05/06 11:43:57 WARNING mlflow.tensorflow: You are saving a TensorFlow Core model or Keras model without a signature. Inference with mlflow.pyfunc.spark_udf() will not work unless the model's pyfunc representation accepts pandas DataFrames as inference inputs.
2026/05/06 11:44:18 WARNING mlflow.utils.environment: Encountered an unexpected error while inferring pip requirements (model URI: C:\Users\Dell\AppData\Local\Temp\tmp77k5hqdm\model, flavor: tensorflow). Fall back to return ['tensorflow==2.21.0', 'cloudpickle==3.1.2']. Set logging level to DEBUG to see the full traceback. 


🏃 View run 3.2_TextCNN at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2/runs/3cce03bdbd504585a361b8b6ad27cb1d
🧪 View experiment at: https://dagshub.com/Oscar-AS/disaster-tweets-project.mlflow/#/experiments/2
